# Stage 4 — Model Comparison & Evaluation
### Pull all three models from the Hub and judge them side by side

**Notebook 4 of 4.** This is the notebook that answers the only question that
really matters:

> **Did any of this actually work?**

Training loss falling is *not* proof. Loss says "the model predicted my training
text well" — it says nothing about whether an answer is correct, useful, or
professional. The only honest test is putting the models side by side on the same
questions.

### What we compare

| Model | What it should look like |
|---|---|
| **Stage 1** (domain pretrained) | Knows HR words. **Can't answer** — rambles or continues the question |
| **Stage 2** (SFT) | **Answers** properly. May be plain, generic, or inconsistent |
| **Stage 3** (DPO, final) | Answers **and** sounds specific and professional |

If Stage 3 doesn't beat Stage 2, DPO didn't help — and it's better to know that
here than in production.

### A note on memory (this is the production-relevant bit)

Three 1.5B models will **not** fit in a free T4's VRAM simultaneously. So we
**load one, generate every answer, free it completely, then load the next.**
It's slower, but it's the only approach that actually runs — and it's how you'd
structure a batch evaluation job anyway.

> **Runtime:** T4 GPU. Expect ~5-10 minutes.

## 1. Install libraries

In [1]:
# ============================================================
# Step 1. Install libraries
# ============================================================
!pip install -q unsloth
!pip install -q transformers datasets peft bitsandbytes accelerate sentencepiece protobuf huggingface_hub pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 MB 14.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 119.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.

## 2. Imports

In [2]:
# ============================================================
# Step 2. Imports
# ============================================================
import torch, gc, json, time
import pandas as pd
from unsloth import FastLanguageModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Device: cuda
GPU: Tesla T4


## Hugging Face login

Every stage pushes its merged model to the Hub, and the next stage pulls it back
down. So we log in once, here, at the top.

**Never paste a raw token into a notebook cell.** A token in a saved `.ipynb` is
a leaked credential — anyone with the file can push or delete on your account.
Use a **Colab secret** instead:

> Click the **key icon (🔑)** in the Colab left sidebar → **Add new secret** →
> Name: `HF_TOKEN`, Value: your token from
> https://huggingface.co/settings/tokens (needs **write** access) → toggle
> **Notebook access** on.

The cell below reads that secret automatically, and falls back to an interactive
prompt if it isn't set.

In [3]:
# ============================================================
# Hugging Face login
# ============================================================
from huggingface_hub import login, whoami

try:
    # Colab secret named HF_TOKEN (recommended).
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    # Fallback: paste the token when prompted.
    login()

HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Logged in as: Mohan143


## 3. Configuration

The three repos we pushed, plus the **evaluation question set**.

**Rule for a fair test:** these questions should be ones the model has *not* seen
verbatim in training. If you evaluate on training questions, you're measuring
memorization, not learning.

In [4]:
# ============================================================
# Step 3. Configuration
# ============================================================
MODELS = {
    "Stage 1 (Domain)": f"{HF_USERNAME}/hr-policy-assistant-stage1",
    "Stage 2 (SFT)":    f"{HF_USERNAME}/hr-policy-assistant-stage2",
    "Stage 3 (DPO)":    f"{HF_USERNAME}/hr-policy-assistant-final",
}

# Held-out evaluation questions - keep this list FIXED across all runs so
# results stay comparable over time.
TEST_QUESTIONS = [
    "How many casual leaves can I take per year?",
    "What is the work from home policy?",
    "How do I apply for sick leave?",
    "What benefits does the company provide?",
    "What is the notice period for resignation?",
    "Can I carry forward my unused casual leave?",
    "What happens if I am consistently late to work?",
]

SYSTEM_PROMPT = "You are a helpful HR Policy Assistant."

# Generation settings - identical for every model, or the comparison is unfair.
MAX_NEW_TOKENS = 150
TEMPERATURE    = 0.7
TOP_P          = 0.9
SEED           = 42

for name, repo in MODELS.items():
    print(f"{name:20s} -> {repo}")
print(f"\n{len(TEST_QUESTIONS)} evaluation questions.")

Stage 1 (Domain)     -> Mohan143/hr-policy-assistant-stage1
Stage 2 (SFT)        -> Mohan143/hr-policy-assistant-stage2
Stage 3 (DPO)        -> Mohan143/hr-policy-assistant-final

7 evaluation questions.


## 4. Helper functions

Two things worth calling out:

- **Identical settings for every model.** Same prompt, same temperature, same seed.
  Change any of those between models and you're comparing the settings, not the
  models.
- **`free_model()` matters.** Deleting the Python variable isn't enough — CUDA
  keeps the memory cached. `torch.cuda.empty_cache()` is what actually releases
  the VRAM so the next model can load.

In [5]:
# ============================================================
# Step 4. Helpers
# ============================================================
def build_prompt(question: str) -> str:
    """Exact same chat template used in Stage 2 and Stage 3 training."""
    return (f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{question}<|im_end|>\n"
            f"<|im_start|>assistant\n")


def generate_answer(model, tokenizer, question: str) -> str:
    """Generate one answer with fixed settings (fair across models)."""
    torch.manual_seed(SEED)   # same seed -> differences come from the model, not luck
    inputs = tokenizer(build_prompt(question), return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            temperature    = TEMPERATURE,
            top_p          = TOP_P,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    # Keep only the assistant's turn.
    return text.split("assistant\n")[-1].strip()


def free_model(*objs):
    """Release VRAM properly - deleting the variable alone is NOT enough."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        print(f"   VRAM after cleanup: {used:.2f} GB")

## 5. Run the evaluation

One model at a time: load → answer all questions → free → next. The results go
into a single table.

In [6]:
# ============================================================
# Step 5. Sequential evaluation (one model in memory at a time)
# ============================================================
results = {}   # {model_name: {question: answer}}

for name, repo in MODELS.items():
    print("=" * 80)
    print(f"Loading {name}  <-  {repo}")
    print("=" * 80)
    t0 = time.time()

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name     = repo,
            max_seq_length = 512,
            dtype          = None,
            load_in_4bit   = True,     # 4-bit so each model fits comfortably
        )
        FastLanguageModel.for_inference(model)
        print(f"Loaded in {time.time()-t0:.0f}s. Generating {len(TEST_QUESTIONS)} answers...")

        answers = {}
        for i, q in enumerate(TEST_QUESTIONS, 1):
            answers[q] = generate_answer(model, tokenizer, q)
            print(f"   [{i}/{len(TEST_QUESTIONS)}] done")
        results[name] = answers

        free_model(model, tokenizer)
        print(f"{name} finished and unloaded.\n")

    except Exception as e:
        print(f"FAILED for {name}: {e}\n")
        results[name] = {q: f"[ERROR: {e}]" for q in TEST_QUESTIONS}

print("All models evaluated.")

Loading Stage 1 (Domain)  <-  Mohan143/hr-policy-assistant-stage1
==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded in 105s. Generating 7 answers...


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

   [1/7] done


...

   [4/7] done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [5/7] done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [6/7] done
   [7/7] done
   VRAM after cleanup: 1.21 GB
Stage 3 (DPO) finished and unloaded.

All models evaluated.


## 6. Side-by-side comparison

Read these **question by question**, across the three models. That vertical
comparison is the whole point — you're looking for a visible progression from
"can't answer" → "answers" → "answers well".

In [7]:
# ============================================================
# Step 6. Print the three-way comparison
# ============================================================
for q in TEST_QUESTIONS:
    print("=" * 100)
    print(f"QUESTION: {q}")
    print("=" * 100)
    for name in MODELS:
        answer = results.get(name, {}).get(q, "[no result]")
        print(f"\n--- {name} ---")
        print(answer[:400] + ("..." if len(answer) > 400 else ""))
    print()

QUESTION: How many casual leaves can I take per year?

--- Stage 1 (Domain) ---
Can I avail leave for my assistant if they are not available during my scheduled leave? If the assistant is not available, can I extend my leave duration accordingly? If the assistant is on leave, can I request extended leave as per policy? The policy allows up to 5 days of casual leave per calendar year. Casual leave can be availed only when the employee's schedule is affected by unforeseen circu...

--- Stage 2 (SFT) ---
If casual leave is exhausted, employees must apply for leave through the HR portal within 24 hours of the intended date. Any unused casual leave must be converted to earned leave for the year. Unused casual leave may be forfeited if not converted within 90 days. Conversion of casual leave to earned leave reduces earned leave balance by the same amount

--- Stage 3 (DPO) ---
The annual casual leave allowance is 5 days per year. Casual leave is not available for employees working in critica

## 7. Comparison table

Same data, in a `DataFrame` you can scan, sort, or export.

In [8]:
# ============================================================
# Step 7. Build a comparison DataFrame
# ============================================================
rows = []
for q in TEST_QUESTIONS:
    row = {"Question": q}
    for name in MODELS:
        row[name] = results.get(name, {}).get(q, "")
    rows.append(row)

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 220)
df

,Question,Stage 1 (Domain),Stage 2 (SFT),Stage 3 (DPO)
0,How many casual leaves can I take per year?,"Can I avail leave for my assistant if they are not available during my scheduled leave? If the assistant is not available, can I extend my leave duration accordingly? If the assistant is on leave, can I request exten...","If casual leave is exhausted, employees must apply for leave through the HR portal within 24 hours of the intended date. Any unused casual leave must be converted to earned leave for the year. Unused casual leave may...",The annual casual leave allowance is 5 days per year. Casual leave is not available for employees working in critical production or emergency situations where immediate attendance is required. Requests must be submit...
1,What is the work from home policy?,The work from home policy allows eligible employees to work from their home office or designated alternative location for specified periods based on business requirements and employee flexibility. This policy require...,"The company supports flexible work arrangements with WFH (Work From Home) option for eligible employees with 5+ years of service. The policy covers: designated WFH days, flexible hours (9-7), one remote day per week,...","The work from home policy allows eligible employees to opt for flexible working hours within specific parameters: employees must have completed at least 2 years of service, have received Manager approval, and be assi..."
2,How do I apply for sick leave?,"The assistant manager will review the leave application and provide feedback within 2 working days. If the leave is approved, the assistant manager will forward the leave request to the HR department for payroll proc...",What is the,"To apply for sick leave, submit a documented sick leave request through the HR portal with your doctor's diagnosis and prescription (if available). The system automatically assigns sick leave days based on the prescr..."
3,What benefits does the company provide?,"Yes, the company offers an online retirement planning tool allowing employees to estimate their retirement needs, calculate required savings, and track progress. The tool provides","Female employees on post-natal leave (26 weeks) receive full salary, return-to-work leave (2 weeks), and breastfeeding support (2 hours/day for 2 months). Employees returning from 2 weeks to 6 months post-natal leave...","The company offers comprehensive benefits including: health insurance (basic and voluntary), life insurance (1x income replacement), accident and critical illness insurance, PF (12% of basic), gratuity (15 years of s..."
4,What is the notice period for resignation?,What are the policy compliance requirements for remote work?裎\n裎policycompliance\nHow is employee data protected during employment?裎\n裎dataprotection\nWhat are the policy compliance requirements for workplace safety?...,Performance bonuses are calculated based on individual performance reviews at the end of each quarter. Bonus payouts range from 0% to 400% of base salary depending on performance ratings. Late bonus submissions may r...,"The standard notice period for resignation is 30 days for roles with 2+ years, 60 days for roles with 1-2 years, and 90 days for roles with 0-1 year. The notice period is calculated from the date of resignation reque..."
5,Can I carry forward my unused casual leave?,"Can I avail leave for my assistant if they are not available during my scheduled leave? If so, how is the leave carried forward? If not, what are the consequences? Can leave be carried forward for assistant leave? If...","Request sick leave through the HR portal, mobile app, or by calling the HR helpdesk. You must declare your symptoms and duration clearly. If you need personal leave, it's typically processed within 24 hours. For medi...","Yes, unused casual leave may be carried forward for up to 12 months with the following conditions: 1) You must have completed at least 12 months of continuous employment with the 

## 8. Automatic signals (useful, but not the verdict)

These are **cheap proxies**, not quality measures. A long answer isn't a good
answer; a short one isn't a bad one. What they're genuinely good at is catching
**failure modes** — a model that's degenerated will show up here as extreme
repetition or a wildly different length.

| Signal | What it hints at |
|---|---|
| Answer length | Stage 1 often rambles to the token limit; SFT/DPO should stop naturally |
| Repetition ratio | High = the model is looping (a classic broken-model symptom) |
| Empty/error rate | Should be zero |

Treat a red flag here as "go look at the actual text", never as a score.

In [9]:
# ============================================================
# Step 8. Cheap automatic signals (diagnostics, NOT quality scores)
# ============================================================
def repetition_ratio(text: str) -> float:
    """Fraction of repeated words. High values suggest the model is looping."""
    words = text.lower().split()
    if len(words) < 5:
        return 0.0
    return 1 - (len(set(words)) / len(words))


stats_rows = []
for name in MODELS:
    answers = list(results.get(name, {}).values())
    answers = [a for a in answers if not a.startswith("[ERROR")]
    if not answers:
        continue
    stats_rows.append({
        "Model": name,
        "Avg words": round(sum(len(a.split()) for a in answers) / len(answers), 1),
        "Avg chars": round(sum(len(a) for a in answers) / len(answers), 1),
        "Avg repetition": round(sum(repetition_ratio(a) for a in answers) / len(answers), 3),
        "Empty answers": sum(1 for a in answers if len(a.strip()) < 5),
    })

pd.DataFrame(stats_rows)

,Model,Avg words,Avg chars,Avg repetition,Empty answers
0,Stage 1 (Domain),94.9,666.9,0.417,0
1,Stage 2 (SFT),62.6,405.7,0.221,0
2,Stage 3 (DPO),87.7,555.6,0.278,0


## 9. Human scoring — the part that actually decides

This is the real evaluation. Everything above just organizes the evidence.

Score each answer **1–5** on three axes:

| Axis | Question to ask |
|---|---|
| **Correctness** | Does it match the actual HR policy? (the one that matters most) |
| **Relevance** | Did it answer *this* question, or drift? |
| **Professionalism** | Would you send this to an employee unedited? |

Run the cell below to produce a blank scoring sheet, fill it in, and average by
model. **A domain expert must do this.** A fluent, confident, wrong answer is the
most dangerous output a model can produce, and *no automatic metric in this
notebook will catch it.*

In [10]:
# ============================================================
# Step 9. Export a blank human-scoring sheet
# ============================================================
score_rows = []
for q in TEST_QUESTIONS:
    for name in MODELS:
        score_rows.append({
            "Question": q,
            "Model": name,
            "Answer": results.get(name, {}).get(q, "")[:500],
            "Correctness (1-5)": "",
            "Relevance (1-5)": "",
            "Professionalism (1-5)": "",
            "Notes": "",
        })

scoring_df = pd.DataFrame(score_rows)
scoring_df.to_csv("human_evaluation_sheet.csv", index=False)
print(f"Wrote human_evaluation_sheet.csv  ({len(scoring_df)} rows to score)")
print("Download it, have an HR expert fill it in, then average by model.")
scoring_df.head(6)

Wrote human_evaluation_sheet.csv  (21 rows to score)
Download it, have an HR expert fill it in, then average by model.


,Question,Model,Answer,Correctness (1-5),Relevance (1-5),Professionalism (1-5),Notes
0,How many casual leaves can I take per year?,Stage 1 (Domain),"Can I avail leave for my assistant if they are not available during my scheduled leave? If the assistant is not available, can I extend my leave duration accordingly? If the assistant is on leave, can I request exten...",,,,
1,How many casual leaves can I take per year?,Stage 2 (SFT),"If casual leave is exhausted, employees must apply for leave through the HR portal within 24 hours of the intended date. Any unused casual leave must be converted to earned leave for the year. Unused casual leave may...",,,,
2,How many casual leaves can I take per year?,Stage 3 (DPO),The annual casual leave allowance is 5 days per year. Casual leave is not available for employees working in critical production or emergency situations where immediate attendance is required. Requests must be submit...,,,,
3,What is the work from home policy?,Stage 1 (Domain),The work from home policy allows eligible employees to work from their home office or designated alternative location for specified periods based on business requirements and employee flexibility. This policy require...,,,,
4,What is the work from home policy?,Stage 2 (SFT),"The company supports flexible work arrangements with WFH (Work From Home) option for eligible employees with 5+ years of service. The policy covers: designated WFH days, flexible hours (9-7), one remote day per week,...",,,,
5,What is the work from home policy?,Stage 3 (DPO),"The work from home policy allows eligible employees to opt for flexible working hours within specific parameters: employees must have completed at least 2 years of service, have received Manager approval, and be assi...",,,,


## 10. Save the raw results

Keep every evaluation run. Version-to-version comparison is how you prove Stage N+1
was actually an improvement — and it's the audit trail a production system needs.

In [11]:
# ============================================================
# Step 10. Persist the raw comparison
# ============================================================
output = {
    "models": MODELS,
    "questions": TEST_QUESTIONS,
    "generation_settings": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "seed": SEED,
    },
    "results": results,
}

with open("model_comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

df.to_csv("model_comparison_table.csv", index=False)
print("Saved: model_comparison_results.json")
print("Saved: model_comparison_table.csv")
print("Saved: human_evaluation_sheet.csv")

Saved: model_comparison_results.json
Saved: model_comparison_table.csv
Saved: human_evaluation_sheet.csv


## How to read your results

### What good looks like

| Model | Expected behaviour |
|---|---|
| **Stage 1** | Uses HR vocabulary but doesn't answer — continues, rambles, or restates the question. **This is correct**, not a bug: it was never taught to answer. |
| **Stage 2** | Answers the question and stops. Possibly plain or generic. **Big visible jump from Stage 1.** |
| **Stage 3** | Same correctness as Stage 2, but more specific and professional. **A subtler jump** — DPO polishes, it doesn't add knowledge. |

### If something looks wrong

| Symptom | Likely cause |
|---|---|
| Stage 2 no better than Stage 1 | Chat template mismatch between training and inference (check it character-for-character), too few examples, or LR too low |
| Stage 3 identical to Stage 2 | `beta` too high, LR too low, or chosen/rejected pairs too similar to teach anything |
| Stage 3 **worse** than Stage 2 — rambling, broken | Over-optimized: `beta` too low or LR too high. **Ship Stage 2 instead.** |
| All models repeat themselves | Overfitting — too many epochs on too little data |
| Answers are confident but factually wrong | The most dangerous outcome. More/better data, and **human review before any deployment** |

### The production takeaway

**A later stage is not automatically better.** Stage 3 *can* be worse than Stage 2
— DPO optimizes for "preferred", which is not the same as "correct". That's the
entire reason this notebook exists.

Ship whichever model wins **this** comparison, judged by a human — not whichever
one is furthest down the pipeline.

---

## Pipeline complete

```text
Stage 1  learned the LANGUAGE     (self-supervised, cross-entropy loss)
Stage 2  learned to ANSWER        (supervised,      cross-entropy loss)
Stage 3  learned WHICH answers    (preference,      DPO loss)
Stage 4  proved whether it worked (human judgement)
```

**Before deploying anything:** have an HR expert sign off on the scoring sheet,
keep a human in the loop for employee-facing answers, and re-run this exact
comparison whenever you retrain.